# Fixpoint — HD-модель из пакета (Google Colab)

Блокнот обучает фотореалистичную 3D-модель (Gaussian splatting) из HD-пакета Fixpoint на бесплатной видеокарте Google.

**Перед началом:** меню **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить.**

**Как пользоваться:**
1. Положите ZIP-пакет (`Fixpoint-HD-….zip`) на Google Диск — в любую папку.
2. Нажмите **Среда выполнения → Выполнить все** (или по очереди ▶ у каждого блока).
3. На шаге 2 разрешите доступ к Google Диску.
4. Через 20–40 минут готовая модель `….ply` появится на Диске рядом с пакетом. Её загружают в приложение: «Ещё» → «HD-скан» → «Загрузить модель».

Модель остаётся в координатах скана и сама встаёт на схему в 3D.

## 1. Видеокарта
Должно быть написано **Tesla T4** (или другая NVIDIA). Если ошибка — включите T4 GPU в меню «Среда выполнения».

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Диск и выбор пакета
Оставьте поле пустым — возьмётся самый свежий `Fixpoint-HD-*.zip` на вашем Диске. Или впишите имя файла.

In [ ]:
#@title Пакет { display-mode: "form" }
ZIP_NAME = ""  #@param {type:"string"}
STEPS = 30000  #@param {type:"integer"}

from google.colab import drive
drive.mount('/content/drive')

import glob, os
zips = [z for z in glob.glob('/content/drive/MyDrive/**/Fixpoint-HD-*.zip', recursive=True)]
if ZIP_NAME.strip():
    zips = [z for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True) if os.path.basename(z) == ZIP_NAME.strip()]
if not zips:
    raise SystemExit('Пакет не найден на Диске. Проверьте, что ZIP загружен на Google Диск.')
ZIP = max(zips, key=os.path.getmtime)
print('Пакет:', ZIP, f'({os.path.getsize(ZIP)/1048576:.1f} МБ)')
if len(zips) > 1 and not ZIP_NAME.strip():
    print('Другие пакеты на Диске:', *[os.path.basename(z) for z in zips if z != ZIP], sep='\n  ')

## 3. Установка Nerfstudio
Занимает 3–8 минут. Предупреждения про версии пакетов можно игнорировать.

In [ ]:
!pip -q install nerfstudio 2>&1 | tail -n 3
!ns-train --help > /dev/null && echo 'Nerfstudio установлен'

## 4. Распаковка

In [ ]:
import zipfile, shutil, json
DATA = '/content/data'
shutil.rmtree(DATA, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DATA)
# transforms.json может лежать в корне или во вложенной папке
tj = glob.glob(DATA + '/**/transforms.json', recursive=True)
if not tj:
    raise SystemExit('В пакете нет transforms.json — это не HD-пакет Fixpoint?')
DATA = os.path.dirname(tj[0])
meta = json.load(open(tj[0]))
# кадры iPhone уже без искажений: модель камеры PINHOLE — Nerfstudio не тратит время и память на выравнивание
meta['camera_model'] = 'PINHOLE'
for k in ('k1', 'k2', 'p1', 'p2'): meta.pop(k, None)
json.dump(meta, open(tj[0], 'w'), indent=1)
print('Кадров:', len(meta['frames']), '| стартовое облако точек:', 'есть' if os.path.exists(os.path.join(DATA, 'points.ply')) else 'нет (старт со случайных точек)')

## 4½. Сборка модуля видеокарты (gsplat)
Один раз за сеанс, 10–20 минут. **Без ограничения `MAX_JOBS` компилятор CUDA запускает много потоков, каждый берёт 1–2 ГБ памяти, оперативная память Colab (12,7 ГБ) кончается, и обучение молча убивается (код 137).** Если блок напишет `True` — модуль собран.

In [ ]:
!MAX_JOBS=2 python3 -c 'from gsplat.cuda._backend import _C; print(_C is not None)' 2>&1 | tail -n 3

## 5. Обучение
На T4 — примерно 20–40 минут на 30 000 шагов. Первый запуск дольше: собирается модуль видеокарты (gsplat).

Флаги `--orientation-method none --center-method none --auto-scale-poses False` оставляют модель в координатах скана — так она встанет на план в приложении. `--downscale-factor 1` — кадры не уменьшаются (и Nerfstudio не задаёт вопрос, на котором блокнот завис бы).

`--pipeline.datamanager.cache-images-type uint8` — кадры хранятся в памяти по байту на цвет: без этого Nerfstudio держит их как float32 (≈ 5 ГБ на 150 кадров 1920×1440), оперативная память Colab (12,7 ГБ) кончается, и процесс молча убивается.

In [ ]:
OUT = '/content/outputs'
shutil.rmtree(OUT, ignore_errors=True)
!MAX_JOBS=2 ns-train splatfacto \
  --output-dir {OUT} \
  --max-num-iterations {STEPS} \
  --viewer.quit-on-train-completion True \
  --vis tensorboard \
  --pipeline.datamanager.cache-images-type uint8 \
  nerfstudio-data --data {DATA} \
  --orientation-method none --center-method none --auto-scale-poses False \
  --downscale-factor 1

## 6. Экспорт и сохранение на Диск
Готовая модель ляжет рядом с пакетом: `<имя пакета>.ply`.

In [ ]:
cfg = sorted(glob.glob(OUT + '/**/config.yml', recursive=True), key=os.path.getmtime)
if not cfg:
    raise SystemExit('Обучение не завершилось — посмотрите ошибку в шаге 5.')
EXP = '/content/export'
shutil.rmtree(EXP, ignore_errors=True)
!ns-export gaussian-splat --load-config {cfg[-1]} --output-dir {EXP}
ply = glob.glob(EXP + '/*.ply')[0]
dst = os.path.splitext(ZIP)[0] + '.ply'
shutil.copy(ply, dst)
print('Готово:', dst, f'({os.path.getsize(dst)/1048576:.1f} МБ)')
print('Загрузите его в приложение: «Ещё» → «HD-скан» → «Загрузить модель».')

Хотите сразу скачать файл на этот компьютер или телефон — запустите блок ниже.

In [ ]:
from google.colab import files
files.download(dst)